# __📊SQL WORK ENVIROMENT. E-COMMERCE FUNNEL DATA ANALYSIS 🛍️🧑‍💻__
---


In [ ]:
#@title 🛢️📊 Conectémonos a nuestra base de datos y activemos nuestro ambiente 🚀 { display-mode: "form" }
import pandas as pd
import sqlite3
from tqdm.notebook import tqdm
from IPython.core.magic import register_cell_magic
from IPython.display import display, HTML, clear_output

# Configuración de visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)

# 1. Configuración de enlaces (Datasets Sincronizados en GitHub)
datasets = {
    'ecommerce_events': "https://media.githubusercontent.com/media/hector1994/e_commerce_funnel_data_generator/refs/heads/master/data_clase_sql_final_mejorado.csv"
}

# 2. Conexión a la base de datos en memoria
connector = sqlite3.connect(':memory:', check_same_thread=False)

# 3. Proceso de carga con Barra de Progreso
print("⬇️🗂️ Descargando y sincronizando Datasets (Hechos + Dimensiones).....🔄⚙️")
summary_data = []

for name, url in tqdm(datasets.items(), desc="Cargando Tablas"):
    try:
        # Descarga de datos
        df = pd.read_csv(url)

        # Conversión de fechas para eventos y clientes
        if name == 'ecommerce_events' and 'event_timestamp' in df.columns:
            df['event_timestamp'] = pd.to_datetime(df['event_timestamp'])

        if name == 'dim_customers' and 'registration_date' in df.columns:
            df['registration_date'] = pd.to_datetime(df['registration_date'])

        # Carga a SQLite
        df.to_sql(name, connector, index=False, if_exists='replace', chunksize=10000)

        summary_data.append({
            "Table Name": f"{name}",
            "Rows": f"{len(df):,}",
            "Columns": len(df.columns)
        })
    except Exception as e:
        print(f"❌ Error cargando {name}: {e}")

# 4. Definición de la "Palabra mágica" %%sql
@register_cell_magic
def sql(line, cell):
    try:
        resultado = pd.read_sql(cell, connector)
        clear_output(wait=True)
        display(HTML("<b style='color: #4CAF50;'>✅ Query completed successfully:</b>"))
        return display(resultado)
    except Exception as e:
        clear_output(wait=True)
        display(HTML(f"<b style='color: #F44336;'>❌ Query execution failed:</b><br><code style='color: grey;'>{str(e)}</code>"))

# 5. Interfaz final
clear_output()
display(HTML("<h2 style='color: #8e62f3'>✨ Database initialized: Full Star Schema ✨</h2>"))
display(HTML("<p>Tables ready: <b>ecommerce_events</b>, <b>dim_products</b>, and <b>dim_customers</b>.</p>"))

display(pd.DataFrame(summary_data))
print("\n ✨🚀 SYSTEM READY! START ANALYZING THE FUNNEL! 🚀✨")

,Table Name,Rows,Columns
0,ecommerce_events,"2,100,000",12



 ✨🚀 SYSTEM READY! START ANALYZING THE FUNNEL! 🚀✨


#### 🚀__QUERY 1__: Generación de métricas de retención en base a queries escalares - respecto al evento anterior.

* __Observación 1:__ Estas métricas con calculadas como KPIs abosulutos (conteo de totales) y relativos (porcentajes) respecto al evento inmediatamente anterior.

* __Observación 2:__ Las queries escalares nos permiten generar una estructura donde cada campo generado puede ser manipulado y definido de forma indepndiente y puntual, lo cual es últil al querer poner métricas asociadas a distintos eventos juntas, sin embargo, por la naturaleza de estas consultas, la siguiente estructura no permite hacer agrupaciones (<code>GROUP BY</code>) por lo que no es recomendada para <code>análisis de cohortes</code>, donde la comparación de las métricas para distintos grupos es el objetvo.

* __Observación 3:__ El uso de <code>Common Table Expressions (CTE)</code> permite generar compartimientos en forma de tablas temporales y poder encapsular eventos puntuales y manipularlos de forma "separada" de la tabla de eventos. Esto permite realizar cálculos de: usuarios, sesiones, ordenes, etc; por evento de forma cómoda. Además, una vez utilizadas y creada una tabla sumario, podemos también almacenar este resultado en otra CTE y así tenerlo "a mano" de forma sencilla, por ejemplo, para exportarlo a Google Sheets y poder hacer un gráfico particular en base a data procesada.

* __Observación 4:__ Notar que en el caso de las métricas relativas de retención, se aplican dos funciones importante fuera del cáculo regular de la métrica en sí para la retención entre etapas __A__ y __B__.

$$100 \times \left[ \frac{\text{Usuarios únicos en evento B}}{\text{Usuarios únicos en evento A}} \right]$$



1. Se aplica primero es <code>NULLIF(cálculo,0)</code> en el denominador de la división, lo cual nos asegura que en caso que este sea cero, evitemos el mensaje de error devido al valor indefinido de dicho cociente. En caso de ser cero el denominador, el resultado arrojará como valor <code>NULL</code>.

2. Se aplica la función <code>ROUND(valor,cantidad_de_decimales)</code> para fijar el nivel de precisión del cálculo.

In [ ]:
%%sql

--PASO 1: GENERAR COMMON TABLE EXPRESSIONS POR CADA ETAPA DEL FUNNEL EN BASE A LA TABLA DE EVENTOS.
--        NOS TRAEMOS EN CADA CASO LOS USUARIOS DISTINTOS ÚNICAMENTE, DADO QUE ES LO QUE NOS INTERESA CONTAR PARA
--        EL CÁLCULO DE LAS MÉTRICAS.

WITH

--HOMEPAGE COMMON TABLE EXPRESSION (CTE)
homepage_cte
AS
(
SELECT DISTINCT user_id
FROM ecommerce_events
WHERE event_name = 'home_page'),


--LOGIN COMMON TABLE EXPRESSION (CTE)
login_cte AS (
SELECT DISTINCT user_id
FROM ecommerce_events
WHERE event_name = 'login'),


--PRODUCT VIEW COMMON TABLE EXPRESSION (CTE)
product_view_cte AS (
  SELECT DISTINCT user_id
  FROM ecommerce_events
WHERE event_name = 'product_view'),


--ADD TO CART COMMON TABLE EXPRESSION (CTE)
add_to_cart_cte AS (
  SELECT DISTINCT user_id
  FROM ecommerce_events
WHERE event_name = 'add_to_cart'),


--BEGIN CHECKOUT COMMON TABLE EXPRESSION (CTE)
begin_checkout_cte AS (
  SELECT DISTINCT user_id
  FROM ecommerce_events
WHERE event_name = 'begin_checkout'),


--END CHECKOUT COMMON TABLE EXPRESSION (CTE)
end_checkout_cte AS (
  SELECT DISTINCT user_id
  FROM ecommerce_events
WHERE event_name = 'end_checkout'),


--PURCHASE COMMON TABLE EXPRESSION (CTE)
purchase_cte AS (
  SELECT DISTINCT user_id
  FROM ecommerce_events
WHERE event_name = 'purchase')




--- PASO 2: REPORTE GENERAL DE MÉTRICAS DE RETENCIÓN.
SELECT

---MÉTRICAS ABSOLUTAS: CONTEO TOTAL DE USUARIOS DISTINTOS POR ETAPA DEL FUNNEL.

(SELECT COUNT(*) FROM homepage_cte)       AS home_page_users,         --TOTAL USUARIOS DISTINTOS EN HOMEPAGE.
(SELECT COUNT(*) FROM login_cte)          AS login_users,             --TOTAL USUARIOS DISTINTOS EN LOGIN.
(SELECT COUNT(*) FROM product_view_cte)   AS product_view_users,      --TOTAL USUARIOS DISTINTOS EN PRODUCT VIEW.
(SELECT COUNT(*) FROM add_to_cart_cte)    AS add_to_cart_users,       --TOTAL USUARIOS DISTINTOS EN ADD TO CART.
(SELECT COUNT(*) FROM begin_checkout_cte) AS begin_checkout_users,    --TOTAL USUARIOS DISTINTOS EN BEGIN CHECKOUT.
(SELECT COUNT(*) FROM end_checkout_cte)   AS end_checkout_users,      --TOTAL USUARIOS DISTINTOS EN END CHECKOUT.
(SELECT COUNT(*) FROM purchase_cte)       AS purchase_users,          --TOTAL USUARIOS DISTINTOS EN PURCHASE.


---MÉTRICAS RELATIVAS: RETENCIÓN % RELATIVA AL EVENTO ANTERIOR EN EL FUNNEL.

--RETENCIÓN LOGIN
ROUND(100.00*(SELECT COUNT(*) FROM login_cte)/
NULLIF((SELECT COUNT(*) FROM homepage_cte),0),2)                 AS login_rate_previous,

--RETENCIÓN PRODUCT VIEW
ROUND(100.00*(SELECT COUNT(*) FROM product_view_cte)/
NULLIF((SELECT COUNT(*) FROM login_cte),0),2)                    AS product_view_rate_previous,

--RETENCIÓN ADD TO CART
ROUND(100.00*(SELECT COUNT(*) FROM add_to_cart_cte)/
NULLIF((SELECT COUNT(*) FROM product_view_cte),0),2)             AS add_to_cart_rate_previous,

--RETENCIÓN BEGIN CHECKOUT
ROUND(100.00*(SELECT COUNT(*) FROM begin_checkout_cte)/
NULLIF((SELECT COUNT(*) FROM add_to_cart_cte),0),2)              AS begin_checkout_rate_previous,

--RETENCIÓN END CHECKOUT
ROUND(100.00*(SELECT COUNT(*) FROM end_checkout_cte)/
NULLIF((SELECT COUNT(*) FROM begin_checkout_cte),0),2)           AS end_checkout_rate_previous,

--RETENCIÓN PURCHASE
ROUND(100.00*(SELECT COUNT(*) FROM purchase_cte)/
NULLIF((SELECT COUNT(*) FROM end_checkout_cte),0),2)             AS purchase_rate_previous



,home_page_users,login_users,product_view_users,add_to_cart_users,begin_checkout_users,end_checkout_users,purchase_users,login_rate_previous,product_view_rate_previous,add_to_cart_rate_previous,begin_checkout_rate_previous,end_checkout_rate_previous,purchase_rate_previous
0,89735,88848,87184,64474,61550,59382,58163,99.01,98.13,73.95,95.46,96.48,97.95


#### 🚀__QUERY 2__: Generación de métricas de retención en base a queries escalares - respecto al evento HOMEPAGE

* __Observación 1:__ Estas métricas con calculadas como KPIs abosulutos (conteo de totales) y relativos (porcentajes) respecto al evento <code>homepage</code>. Notar que esta elección es arbitraria, dado que podría ser relativo a cualquier evento, sin embargo, dado que la base inicial de usuarios considera es la que llega al <code>homepage</code>, se toma como punto de referencia.

* __Observación 2:__ Este cambio en el cáculo solo implica cambiar el denominador en cada métrica relativa al conteo de usuarios distintos en el <code>homepage</code>.

In [ ]:
%%sql

--PASO 1: GENERAR COMMON TABLE EXPRESSIONS POR CADA ETAPA DEL FUNNEL EN BASE A LA TABLA DE EVENTOS.
--        NOS TRAEMOS EN CADA CASO LOS USUARIOS DISTINTOS ÚNICAMENTE, DADO QUE ES LO QUE NOS INTERESA CONTAR PARA
--        EL CÁLCULO DE LAS MÉTRICAS.

WITH

--HOMEPAGE COMMON TABLE EXPRESSION (CTE)
homepage_cte
AS
(
SELECT DISTINCT user_id
FROM ecommerce_events
WHERE event_name = 'home_page'),


--LOGIN COMMON TABLE EXPRESSION (CTE)
login_cte AS (
SELECT DISTINCT user_id
FROM ecommerce_events
WHERE event_name = 'login'),


--PRODUCT VIEW COMMON TABLE EXPRESSION (CTE)
product_view_cte AS (
  SELECT DISTINCT user_id
  FROM ecommerce_events
WHERE event_name = 'product_view'),


--ADD TO CART COMMON TABLE EXPRESSION (CTE)
add_to_cart_cte AS (
  SELECT DISTINCT user_id
  FROM ecommerce_events
WHERE event_name = 'add_to_cart'),


--BEGIN CHECKOUT COMMON TABLE EXPRESSION (CTE)
begin_checkout_cte AS (
  SELECT DISTINCT user_id
  FROM ecommerce_events
WHERE event_name = 'begin_checkout'),


--END CHECKOUT COMMON TABLE EXPRESSION (CTE)
end_checkout_cte AS (
  SELECT DISTINCT user_id
  FROM ecommerce_events
WHERE event_name = 'end_checkout'),


--PURCHASE COMMON TABLE EXPRESSION (CTE)
purchase_cte AS (
  SELECT DISTINCT user_id
  FROM ecommerce_events
WHERE event_name = 'purchase')




--- PASO 2: REPORTE GENERAL DE MÉTRICAS DE RETENCIÓN.
SELECT

---MÉTRICAS ABSOLUTAS: CONTEO TOTAL DE USUARIOS DISTINTOS POR ETAPA DEL FUNNEL.

(SELECT COUNT(*) FROM homepage_cte)       AS home_page_users,         --TOTAL USUARIOS DISTINTOS EN HOMEPAGE.
(SELECT COUNT(*) FROM login_cte)          AS login_users,             --TOTAL USUARIOS DISTINTOS EN LOGIN.
(SELECT COUNT(*) FROM product_view_cte)   AS product_view_users,      --TOTAL USUARIOS DISTINTOS EN PRODUCT VIEW.
(SELECT COUNT(*) FROM add_to_cart_cte)    AS add_to_cart_users,       --TOTAL USUARIOS DISTINTOS EN ADD TO CART.
(SELECT COUNT(*) FROM begin_checkout_cte) AS begin_checkout_users,    --TOTAL USUARIOS DISTINTOS EN BEGIN CHECKOUT.
(SELECT COUNT(*) FROM end_checkout_cte)   AS end_checkout_users,      --TOTAL USUARIOS DISTINTOS EN END CHECKOUT.
(SELECT COUNT(*) FROM purchase_cte)       AS purchase_users,          --TOTAL USUARIOS DISTINTOS EN PURCHASE.


---MÉTRICAS RELATIVAS: RETENCIÓN % RELATIVA AL EVENTO HOMEPAGE.

--RETENCIÓN LOGIN
ROUND(100.00*(SELECT COUNT(*) FROM login_cte)/
NULLIF((SELECT COUNT(*) FROM homepage_cte),0),2)                    AS login_rate_base,

--RETENCIÓN PRODUCT VIEW
ROUND(100.00*(SELECT COUNT(*) FROM product_view_cte)/
NULLIF((SELECT COUNT(*) FROM homepage_cte),0),2)                    AS product_view_rate_base,

--RETENCIÓN ADD TO CART
ROUND(100.00*(SELECT COUNT(*) FROM add_to_cart_cte)/
NULLIF((SELECT COUNT(*) FROM homepage_cte),0),2)                    AS add_to_cart_rate_base,

--RETENCIÓN BEGIN CHECKOUT
ROUND(100.00*(SELECT COUNT(*) FROM begin_checkout_cte)/
NULLIF((SELECT COUNT(*) FROM homepage_cte),0),2)                    AS begin_checkout_rate_base,

--RETENCIÓN END CHECKOUT
ROUND(100.00*(SELECT COUNT(*) FROM end_checkout_cte)/
NULLIF((SELECT COUNT(*) FROM homepage_cte),0),2)                    AS end_checkout_rate_base,

--RETENCIÓN PURCHASE
ROUND(100.00*(SELECT COUNT(*) FROM purchase_cte)/
NULLIF((SELECT COUNT(*) FROM homepage_cte),0),2)                    AS purchase_rate_base



,home_page_users,login_users,product_view_users,add_to_cart_users,begin_checkout_users,end_checkout_users,purchase_users,login_rate_base,product_view_rate_base,add_to_cart_rate_base,begin_checkout_rate_base,end_checkout_rate_base,purchase_rate_base
0,89735,88848,87184,64474,61550,59382,58163,99.01,97.16,71.85,68.59,66.17,64.82


#### 🚀__QUERY 3__: Generación de métricas de abandono en base a queries escalares - respecto al evento anterior.

* __Observación 1:__ Notar la que construcción de las métricas es bastante similar. El abandono entre los eventos __A__ y __B__ puede calcularse de forma abosoluta o relativa.

__Abandono neto:__ Cantidad de usuarios únicos que abandonan al ir de la etapa A a la etapa B.
$$ \left( \text{Usuarios únicos en A} - \text{Usuarios únicos en B} \right) $$

__Tasa de abandono (%):__ Porcentaje-razón de usuarios que abandonan al ir de la etapa A a la etapa B.
$$100 \times \left[ \frac{\text{Usuarios únicos en evento A} - \text{Usuarios únicos en evento B}}{\text{Usuarios únicos en evento A}} \right]$$



* __Observación 2:__ Se mantiene el uso de <code>NULLIF</code> y <code>ROUND()</code> para verificar la división por cero y para redondear el número hasta el número de decimales deseados.

* __Observación 3:__ Se deja de tarea realizar estos cálculos para las métricas de abandono respecto al evento inicial <code>homepage</code>. El procedimiento es análogo al de las métricas de retención.

In [ ]:
%%sql

--PASO 1: GENERAR COMMON TABLE EXPRESSIONS POR CADA ETAPA DEL FUNNEL EN BASE A LA TABLA DE EVENTOS.
--        NOS TRAEMOS EN CADA CASO LOS USUARIOS DISTINTOS ÚNICAMENTE, DADO QUE ES LO QUE NOS INTERESA CONTAR PARA
--        EL CÁLCULO DE LAS MÉTRICAS.

WITH

--HOMEPAGE COMMON TABLE EXPRESSION (CTE)
homepage_cte
AS
(
SELECT DISTINCT user_id
FROM ecommerce_events
WHERE event_name = 'home_page'),


--LOGIN COMMON TABLE EXPRESSION (CTE)
login_cte AS (
SELECT DISTINCT user_id
FROM ecommerce_events
WHERE event_name = 'login'),


--PRODUCT VIEW COMMON TABLE EXPRESSION (CTE)
product_view_cte AS (
  SELECT DISTINCT user_id
  FROM ecommerce_events
WHERE event_name = 'product_view'),


--ADD TO CART COMMON TABLE EXPRESSION (CTE)
add_to_cart_cte AS (
  SELECT DISTINCT user_id
  FROM ecommerce_events
WHERE event_name = 'add_to_cart'),


--BEGIN CHECKOUT COMMON TABLE EXPRESSION (CTE)
begin_checkout_cte AS (
  SELECT DISTINCT user_id
  FROM ecommerce_events
WHERE event_name = 'begin_checkout'),


--END CHECKOUT COMMON TABLE EXPRESSION (CTE)
end_checkout_cte AS (
  SELECT DISTINCT user_id
  FROM ecommerce_events
WHERE event_name = 'end_checkout'),


--PURCHASE COMMON TABLE EXPRESSION (CTE)
purchase_cte AS (
  SELECT DISTINCT user_id
  FROM ecommerce_events
WHERE event_name = 'purchase')


-- PASO 2: REPORTE GENERAL DE MÉTRICAS DE ABANDONO.
SELECT

---MÉTRICAS ABSOLUTAS: CONTEO TOTAL DE USUARIOS DISTINTOS POR ETAPA DEL FUNNEL.

(SELECT COUNT(*) FROM homepage_cte)       AS home_page_users,         --TOTAL USUARIOS DISTINTOS EN HOMEPAGE.
(SELECT COUNT(*) FROM login_cte)          AS login_users,             --TOTAL USUARIOS DISTINTOS EN LOGIN.
(SELECT COUNT(*) FROM product_view_cte)   AS product_view_users,      --TOTAL USUARIOS DISTINTOS EN PRODUCT VIEW.
(SELECT COUNT(*) FROM add_to_cart_cte)    AS add_to_cart_users,       --TOTAL USUARIOS DISTINTOS EN ADD TO CART.
(SELECT COUNT(*) FROM begin_checkout_cte) AS begin_checkout_users,    --TOTAL USUARIOS DISTINTOS EN BEGIN CHECKOUT.
(SELECT COUNT(*) FROM end_checkout_cte)   AS end_checkout_users,      --TOTAL USUARIOS DISTINTOS EN END CHECKOUT.
(SELECT COUNT(*) FROM purchase_cte)       AS purchase_users,          --TOTAL USUARIOS DISTINTOS EN PURCHASE.


--MÉTRICA ABSOLUTAS-ABANDONO ABSOLUTO: DIFERENCIA NETA DE USUARIOS EN UNA ETAPA A Y LOS USUARIOS EN LA ETAPA B.

--ABANDONO LOGIN - ABSOLUTO
((SELECT COUNT(*) FROM homepage_cte)        - (SELECT COUNT(*) FROM login_cte))           AS login_absolute_dropoff,
--ABANDONO PRODUCT VIEW - ABSOLUTO
((SELECT COUNT(*) FROM login_cte)           - (SELECT COUNT(*) FROM product_view_cte))    AS product_view_absolute_dropoff,
--ABANDONO ADD TO CART - ABSOLUTO
((SELECT COUNT(*) FROM product_view_cte)    - (SELECT COUNT(*) FROM add_to_cart_cte))     AS add_to_cart_absolute_dropoff,
--ABANDONO BEGIN CHECKOUT - ABSOLUTO
((SELECT COUNT(*) FROM add_to_cart_cte)     - (SELECT COUNT(*) FROM begin_checkout_cte))  AS begin_checkout_absolute_dropoff,
--ABANDONO END CHECKOUT - ABSOLUTO
((SELECT COUNT(*) FROM begin_checkout_cte)  - (SELECT COUNT(*) FROM end_checkout_cte))    AS end_checkout_absolute_dropoff,
--ABANDONO PURCHASE - ABSOLUTO
((SELECT COUNT(*) FROM end_checkout_cte)    - (SELECT COUNT(*) FROM purchase_cte))        AS purchase_absolute_dropoff,


--MÉTRICAS RELATIVAS: % ABANDONO RELATIVO AL EVENTO ANTERIOR.

--ABANDONO LOGIN - RATE
ROUND(100.00*((SELECT COUNT(*) FROM homepage_cte) - (SELECT COUNT(*) FROM login_cte))/
NULLIF((SELECT COUNT(*) FROM homepage_cte),0),2)                                                AS login_dropoff_rate,

--ABANDONO PRODUCT_VIEW - RATE
ROUND(100.00*((SELECT COUNT(*) FROM login_cte) - (SELECT COUNT(*) FROM product_view_cte))/
NULLIF((SELECT COUNT(*) FROM login_cte),0),2)                                                   AS product_view_dropoff_rate,

--ABANDONO ADD TO CART - RATE
ROUND(100.00*((SELECT COUNT(*) FROM product_view_cte) - (SELECT COUNT(*) FROM add_to_cart_cte))/
NULLIF((SELECT COUNT(*) FROM product_view_cte),0),2)                                            AS add_to_cart_dropoff_rate,

--ABANDONO BEGIN CHECKOUT - RATE
ROUND(100.00*((SELECT COUNT(*) FROM add_to_cart_cte) - (SELECT COUNT(*) FROM begin_checkout_cte))/
NULLIF((SELECT COUNT(*) FROM add_to_cart_cte),0),2)                                             AS begin_checkout_dropoff_rate,

--ABANDONO END CHECKOUT - RATE
ROUND(100.00*((SELECT COUNT(*) FROM begin_checkout_cte) - (SELECT COUNT(*) FROM end_checkout_cte))/
NULLIF((SELECT COUNT(*) FROM begin_checkout_cte),0),2)                                          AS end_checkout_dropoff_rate,

--ABANDONO PRUCHASE - RATE
ROUND(100.00*((SELECT COUNT(*) FROM end_checkout_cte) - (SELECT COUNT(*) FROM purchase_cte))/
NULLIF((SELECT COUNT(*) FROM end_checkout_cte),0),2)                                            AS purchase_dropoff_rate

,home_page_users,login_users,product_view_users,add_to_cart_users,begin_checkout_users,end_checkout_users,purchase_users,login_absolute_dropoff,product_view_absolute_dropoff,add_to_cart_absolute_dropoff,begin_checkout_absolute_dropoff,end_checkout_absolute_dropoff,purchase_absolute_dropoff,login_dropoff_rate,product_view_dropoff_rate,add_to_cart_dropoff_rate,begin_checkout_dropoff_rate,end_checkout_dropoff_rate,purchase_dropoff_rate
0,89735,88848,87184,64474,61550,59382,58163,887,1664,22710,2924,2168,1219,0.99,1.87,26.05,4.54,3.52,2.05


#### 🚀__QUERY 4__: Generación de métricas de retención en base a agregación condicional - respecto al evento anterior.

* __Observación 1:__ De forma alternativa para generar el reporte anterior, podemos usar otra forma de codificación para calcular las mismas cantidades. La forma presentada, se llama <code>agregación condicional</code>.
* __Observación 2:__ La agregación condicional, implica la combinación de las funciones de agregación, como: <code>SUM()</code>, <code>COUNT()</code>, <code>MAX()</code>,<code>MIN()</code>, etc; en conjunto con las sentencia <code>CASE WHEN</code> para usar estas condiciones sobre registros que cumplen una condución en particular, como por ejemplo, estar asociada a un evento específico.

* __Observación 3:__ Fuera de lo anterior, la estructura y cálculos a realizar con análogos y faciles de replicar.

* __Observación 4:__ La ventaja principal de este enfoque, es que se constituye con una estructura idéntica a nuestro esqueléto de consulta habitual, permitiendo agrupaciones, filtros y ordenamientos globales, permitiendo, entre otras cosas, el <code>análisis de cohortes</code>.

In [ ]:
%%sql
SELECT
       country,
       device,

        --AGREGACIÓN CONDICIONAL: CALCULO DE USUARIOS DISTINTOS EN CADA ETAPA DEL FUNNEL (MÉTRICA ABSOLUTA).
        COUNT(DISTINCT CASE WHEN event_name = 'home_page'     THEN user_id END) AS home_page_users,
        COUNT(DISTINCT CASE WHEN event_name = 'login'         THEN user_id END) AS login_users,
        COUNT(DISTINCT CASE WHEN event_name = 'product_view'  THEN user_id END) AS product_view_users,
        COUNT(DISTINCT CASE WHEN event_name = 'add_to_cart'   THEN user_id END) AS add_to_cart_users,
        COUNT(DISTINCT CASE WHEN event_name = 'begin_checkout'THEN user_id END) AS begin_checkout_users,
        COUNT(DISTINCT CASE WHEN event_name = 'end_checkout'  THEN user_id END) AS end_checkout_users,
        COUNT(DISTINCT CASE WHEN event_name = 'purchase'      THEN user_id END) AS purchase_users,


        --CALCULO DE MÉTRICAS DE RETENCIÓN - RESPECTO AL EVENTO ANTERIOR.

        --RETENCIÓN (%) LOGIN:
        ROUND(100.00*COUNT(DISTINCT CASE WHEN event_name = 'login' THEN user_id END)/
        NULLIF(COUNT(DISTINCT CASE WHEN event_name = 'home_page' THEN user_id END),0),2)      AS login_rate_previous,
        --RETENCIÓN (%) PRODUCT VIEW:
        ROUND(100.00*COUNT(DISTINCT CASE WHEN event_name = 'product_view' THEN user_id END)/
        NULLIF(COUNT(DISTINCT CASE WHEN event_name = 'login' THEN user_id END),0),2)          AS product_view_rate_previous,
        --RETENCIÓN (%) ADD TO CART:
        ROUND(100.00*COUNT(DISTINCT CASE WHEN event_name = 'add_to_cart' THEN user_id END)/
        NULLIF(COUNT(DISTINCT CASE WHEN event_name = 'product_view' THEN user_id END),0),2)   AS add_to_cart_rate_previous,
        --RETENCIÓN (%) BEGIN CHECKOUT:
        ROUND(100.00*COUNT(DISTINCT CASE WHEN event_name = 'begin_checkout' THEN user_id END)/
        NULLIF(COUNT(DISTINCT CASE WHEN event_name = 'add_to_cart' THEN user_id END),0),2)    AS begin_checkout_rate_previous,
        --RETENCIÓN (%) END CHECKOUT:
        ROUND(100.00*COUNT(DISTINCT CASE WHEN event_name = 'end_checkout' THEN user_id END)/
        NULLIF(COUNT(DISTINCT CASE WHEN event_name = 'begin_checkout' THEN user_id END),0),2) AS end_checkout_rate_previous,
        --RETENCIÓN (%) PURCHASE:
        ROUND(100.00*COUNT(DISTINCT CASE WHEN event_name = 'purchase' THEN user_id END)/
        NULLIF(COUNT(DISTINCT CASE WHEN event_name = 'end_checkout' THEN user_id END),0),2)   AS purchase_rate_previous


        FROM ecommerce_events
        WHERE DATE(event_timestamp) BETWEEN '2026-01-01' AND '2026-01-15'
        GROUP BY 1,2
        ORDER BY 2 DESC

--

,country,device,home_page_users,login_users,product_view_users,add_to_cart_users,begin_checkout_users,end_checkout_users,purchase_users,login_rate_previous,product_view_rate_previous,add_to_cart_rate_previous,begin_checkout_rate_previous,end_checkout_rate_previous,purchase_rate_previous
0,Argentina,Tablet,2591,2008,1634,599,543,516,502,77.50,81.37,36.66,90.65,95.03,97.29
1,Chile,Tablet,1722,1308,1044,401,363,341,324,75.96,79.82,38.41,90.52,93.94,95.01
2,Colombia,Tablet,3364,2586,2120,774,705,675,654,76.87,81.98,36.51,91.09,95.74,96.89
3,España,Tablet,5512,4277,3543,1374,1264,1186,1146,77.59,82.84,38.78,91.99,93.83,96.63
4,México,Tablet,8694,6694,5530,2135,1959,1840,1780,77.00,82.61,38.61,91.76,93.93,96.74
5,Argentina,Mobile,8829,7777,6881,3350,3085,2902,2821,88.08,88.48,48.68,92.09,94.07,97.21
6,Chile,Mobile,5764,5072,4525,2165,2036,1917,1864,87.99,89.22,47.85,94.04,94.16,97.24
7,Colombia,Mobile,11041,9725,8559,4101,3790,3585,3475,88.08,88.01,47.91,92.42,94.59,96.93
8,España,Mobile,18332,16057,14221,6845,6358,6051,5859,87.59,88.57,48.13,92.89,95.17,96.83
9,México,Mobile,29314,25839,22965,11129,10319,9757,9448,88.15,88.88,48.46,92.72,94.55,96.83


#### 🚀__QUERY 5__: Generación de métricas de retención en base a agregación condicional - respecto al evento homepage.


In [ ]:
%%sql
SELECT

        --AQUI PUEDES ENUMERAR DIMENSIONES COMO: country, device, etc; si deseas agrupar. No olvides
        --                                       declarar estas dimensiones en el GROUP BY para hacer
        --                                       análisis de cohortes.

        --AGREGACIÓN CONDICIONAL: CALCULO DE USUARIOS DISTINTOS EN CADA ETAPA DEL FUNNEL (MÉTRICA ABSOLUTA).
        COUNT(DISTINCT CASE WHEN event_name = 'home_page'     THEN user_id END) AS home_page_users,
        COUNT(DISTINCT CASE WHEN event_name = 'login'         THEN user_id END) AS login_users,
        COUNT(DISTINCT CASE WHEN event_name = 'product_view'  THEN user_id END) AS product_view_users,
        COUNT(DISTINCT CASE WHEN event_name = 'add_to_cart'   THEN user_id END) AS add_to_cart_users,
        COUNT(DISTINCT CASE WHEN event_name = 'begin_checkout'THEN user_id END) AS begin_checkout_users,
        COUNT(DISTINCT CASE WHEN event_name = 'end_checkout'  THEN user_id END) AS end_checkout_users,
        COUNT(DISTINCT CASE WHEN event_name = 'purchase'      THEN user_id END) AS purchase_users,


        --CALCULO DE MÉTRICAS DE RETENCIÓN - RESPECTO AL EVENTO HOMEPAGE.

        --RETENCIÓN (%) LOGIN:
        ROUND(100.00*COUNT(DISTINCT CASE WHEN event_name = 'login' THEN user_id END)/
        NULLIF(COUNT(DISTINCT CASE WHEN event_name = 'home_page' THEN user_id END),0),2)       AS login_rate_base,
        --RETENCIÓN (%) PRODUCT VIEW:
        ROUND(100.00*COUNT(DISTINCT CASE WHEN event_name = 'product_view' THEN user_id END)/
        NULLIF(COUNT(DISTINCT CASE WHEN event_name = 'home_page' THEN user_id END),0),2)       AS product_view_rate_base,
        --RETENCIÓN (%) ADD TO CART:
        ROUND(100.00*COUNT(DISTINCT CASE WHEN event_name = 'add_to_cart' THEN user_id END)/
        NULLIF(COUNT(DISTINCT CASE WHEN event_name = 'home_page' THEN user_id END),0),2)       AS add_to_cart_rate_base,
        --RETENCIÓN (%) BEGIN CHECKOUT:
        ROUND(100.00*COUNT(DISTINCT CASE WHEN event_name = 'begin_checkout' THEN user_id END)/
        NULLIF(COUNT(DISTINCT CASE WHEN event_name = 'home_page' THEN user_id END),0),2)       AS begin_checkout_rate_base,
        --RETENCIÓN (%) END CHECKOUT:
        ROUND(100.00*COUNT(DISTINCT CASE WHEN event_name = 'end_checkout' THEN user_id END)/
        NULLIF(COUNT(DISTINCT CASE WHEN event_name = 'home_page' THEN user_id END),0),2)       AS end_checkout_rate_base,
        --RETENCIÓN (%) PURCHASE:
        ROUND(100.00*COUNT(DISTINCT CASE WHEN event_name = 'purchase' THEN user_id END)/
        NULLIF(COUNT(DISTINCT CASE WHEN event_name = 'home_page' THEN user_id END),0),2)       AS purchase_rate_base


        FROM ecommerce_events

,home_page_users,login_users,product_view_users,add_to_cart_users,begin_checkout_users,end_checkout_users,purchase_users,login_rate_base,product_view_rate_base,add_to_cart_rate_base,begin_checkout_rate_base,end_checkout_rate_base,purchase_rate_base
0,89735,88848,87184,64474,61550,59382,58163,99.01,97.16,71.85,68.59,66.17,64.82


#### 🚀__QUERY 6__: Generación de métricas de abandono en base a agregación condicional - respecto al evento anterior.

In [ ]:
%%sql
SELECT

        DATE(event_timestamp) AS date,

        --AGREGACIÓN CONDICIONAL: CALCULO DE USUARIOS DISTINTOS EN CADA ETAPA DEL FUNNEL (MÉTRICA ABSOLUTA).
        COUNT(DISTINCT CASE WHEN event_name = 'home_page'     THEN user_id END) AS home_page_users,
        COUNT(DISTINCT CASE WHEN event_name = 'login'         THEN user_id END) AS login_users,
        COUNT(DISTINCT CASE WHEN event_name = 'product_view'  THEN user_id END) AS product_view_users,
        COUNT(DISTINCT CASE WHEN event_name = 'add_to_cart'   THEN user_id END) AS add_to_cart_users,
        COUNT(DISTINCT CASE WHEN event_name = 'begin_checkout'THEN user_id END) AS begin_checkout_users,
        COUNT(DISTINCT CASE WHEN event_name = 'end_checkout'  THEN user_id END) AS end_checkout_users,
        COUNT(DISTINCT CASE WHEN event_name = 'purchase'      THEN user_id END) AS purchase_users,


        --CALCULO DE MÉTRICAS DE ABANDONO - RESPECTO AL EVENTO ANTERIOR.
        --ABANDONO ABSOLUTO: DIFERENCIA ENTRE LA CANTIDAD DE USUARIOS ÚNICOS ENTRE LOS EVENTOS A Y B.


        --ABANDONO LOGIN - ABSOLUTO
        COUNT(DISTINCT CASE WHEN event_name = 'home_page' THEN user_id END) -
        COUNT(DISTINCT CASE WHEN event_name = 'login'     THEN user_id END)             AS login_absolute_dropoff,
        --ABANDONO PRODUCT VIEW - ABSOLUTO
        COUNT(DISTINCT CASE WHEN event_name = 'login' THEN user_id END) -
        COUNT(DISTINCT CASE WHEN event_name = 'product_view'     THEN user_id END)       AS product_view_absolute_dropoff,
        --ABANDONO ADD TO CART - ABSOLUTO
        COUNT(DISTINCT CASE WHEN event_name = 'product_view' THEN user_id END) -
        COUNT(DISTINCT CASE WHEN event_name = 'add_to_cart'  THEN user_id END)           AS add_to_cart_absolute_dropoff,
        --ABANDONO BEGIN CHECKOUT - ABSOLUTO
        COUNT(DISTINCT CASE WHEN event_name = 'add_to_cart' THEN user_id END) -
        COUNT(DISTINCT CASE WHEN event_name = 'begin_checkout'     THEN user_id END)     AS begin_checkout_absolute_dropoff,
        --ABANDONO END CHECKOUT - ABSOLUTO
        COUNT(DISTINCT CASE WHEN event_name = 'begin_checkout' THEN user_id END) -
        COUNT(DISTINCT CASE WHEN event_name = 'end_checkout'     THEN user_id END)       AS end_checkout_absolute_dropoff,
        --ABANDONO PURCHASE - ABSOLUTO
        COUNT(DISTINCT CASE WHEN event_name = 'end_checkout' THEN user_id END) -
        COUNT(DISTINCT CASE WHEN event_name = 'purchase'     THEN user_id END)           AS purchase_absolute_dropoff,


        --MÉTRICAS RELATIVAS: % ABANDONO RELATIVO AL EVENTO ANTERIOR.

        --ABANDONO LOGIN: RATE
        ROUND(100.00*(COUNT(DISTINCT CASE WHEN event_name = 'home_page' THEN user_id END) -
        COUNT(DISTINCT CASE WHEN event_name = 'login'     THEN user_id END))

        /NULLIF(COUNT(DISTINCT CASE WHEN event_name = 'home_page' THEN user_id END),0),2)  AS login_dropoff_rate,

        --ABANDONO PRODUCT VIEW: RATE
        ROUND(100.00*(COUNT(DISTINCT CASE WHEN event_name = 'login' THEN user_id END) -
        COUNT(DISTINCT CASE WHEN event_name = 'product_view'     THEN user_id END))

        /NULLIF(COUNT(DISTINCT CASE WHEN event_name = 'login' THEN user_id END),0),2)  AS product_view_dropoff_rate,

       --ABANDONO ADD TO CART: RATE
       ROUND(100.00*(COUNT(DISTINCT CASE WHEN event_name = 'product_view' THEN user_id END) -
       COUNT(DISTINCT CASE WHEN event_name = 'add_to_cart'     THEN user_id END))

       /NULLIF(COUNT(DISTINCT CASE WHEN event_name = 'product_view' THEN user_id END),0),2)  AS add_to_cart_dropoff_rate,

       --ABANDONO BEGIN CHECKOUT: RATE
       ROUND(100.00*(COUNT(DISTINCT CASE WHEN event_name = 'add_to_cart' THEN user_id END) -
       COUNT(DISTINCT CASE WHEN event_name = 'begin_checkout'     THEN user_id END))

       /NULLIF(COUNT(DISTINCT CASE WHEN event_name = 'add_to_cart' THEN user_id END),0),2)  AS begin_checkout_dropoff_rate,

       --ABANDONO END CHECKOUT: RATE
       ROUND(100.00*(COUNT(DISTINCT CASE WHEN event_name = 'begin_checkout' THEN user_id END) -
       COUNT(DISTINCT CASE WHEN event_name = 'end_checkout'     THEN user_id END))

       /NULLIF(COUNT(DISTINCT CASE WHEN event_name = 'begin_checkout' THEN user_id END),0),2)  AS end_checkout_dropoff_rate,

       --ABANDONO PURCHASE: RATE
       ROUND(100.00*(COUNT(DISTINCT CASE WHEN event_name = 'end_checkout' THEN user_id END) -
       COUNT(DISTINCT CASE WHEN event_name = 'purchase'     THEN user_id END))

       /NULLIF(COUNT(DISTINCT CASE WHEN event_name = 'end_checkout' THEN user_id END),0),2)  AS purchase_dropoff_rate

        FROM ecommerce_events GROUP BY 1 ORDER BY 1 ASC

,date,home_page_users,login_users,product_view_users,add_to_cart_users,begin_checkout_users,end_checkout_users,purchase_users,login_absolute_dropoff,product_view_absolute_dropoff,add_to_cart_absolute_dropoff,begin_checkout_absolute_dropoff,end_checkout_absolute_dropoff,purchase_absolute_dropoff,login_dropoff_rate,product_view_dropoff_rate,add_to_cart_dropoff_rate,begin_checkout_dropoff_rate,end_checkout_dropoff_rate,purchase_dropoff_rate
0,2026-01-01,15454,11867,9616,3673,3356,3150,3035,3587,2251,5943,317,206,115,23.21,18.97,61.80,8.63,6.14,3.65
1,2026-01-02,15244,11780,9573,3565,3261,3047,2937,3464,2207,6008,304,214,110,22.72,18.74,62.76,8.53,6.56,3.61
2,2026-01-03,15453,11858,9724,3608,3311,3108,3006,3595,2134,6116,297,203,102,23.26,18.00,62.90,8.23,6.13,3.28
3,2026-01-04,15230,11593,9404,3506,3222,3031,2929,3637,2189,5898,284,191,102,23.88,18.88,62.72,8.10,5.93,3.37
4,2026-01-05,15217,11653,9457,3572,3279,3068,2954,3564,2196,5885,293,211,114,23.42,18.84,62.23,8.20,6.43,3.72
5,2026-01-06,15047,11618,9397,3518,3221,3023,2916,3429,2221,5879,297,198,107,22.79,19.12,62.56,8.44,6.15,3.54
6,2026-01-07,15199,11598,9409,3491,3204,3027,2937,3601,2189,5918,287,177,90,23.69,18.87,62.90,8.22,5.52,2.97
7,2026-01-08,15542,11853,9610,3657,3341,3128,3016,3689,2243,5953,316,213,112,23.74,18.92,61.95,8.64,6.38,3.58
8,2026-01-09,15299,11762,9553,3642,3343,3137,3027,3537,2209,5911,299,206,110,23.12,18.78,61.88,8.21,6.16,3.51
9,2026-01-10,15131,11592,9393,3571,3274,3067,2941,3539,2199,5822,297,207,126,23.39,18.97,61.98,8.32,6.32,4.11


### __Query 7: A/B Testing y ARPU (Average Revenue Per User)__

In [ ]:
%%sql

WITH baseline AS ( ---- A
    SELECT
        device,
        COUNT(DISTINCT CASE WHEN event_name = 'home_page'      THEN user_id END) AS homepage,
        COUNT(DISTINCT CASE WHEN event_name = 'login'          THEN user_id END) AS login,
        COUNT(DISTINCT CASE WHEN event_name = 'product_view'   THEN user_id END) AS product_view,
        COUNT(DISTINCT CASE WHEN event_name = 'add_to_cart'    THEN user_id END) AS add_to_cart,
        COUNT(DISTINCT CASE WHEN event_name = 'begin_checkout' THEN user_id END) AS begin_checkout,
        COUNT(DISTINCT CASE WHEN event_name = 'end_checkout'   THEN user_id END) AS end_checkout,
        COUNT(DISTINCT CASE WHEN event_name = 'purchase'       THEN user_id END) AS purchase
    FROM ecommerce_events GROUP BY 1,2),

simulated_funnel AS (---B: Mejorar en un 20% los usuarios en Add to cart, de modo de minorizar el impacto

SELECT
device,


homepage,
login,
product_view,
ROUND(add_to_cart + (product_view - add_to_cart)*0.2,0)       AS add_to_cart_sim, -- ADD TO CART SIMULADO.
ROUND(begin_checkout + (product_view - add_to_cart)*0.2*(1.0*begin_checkout/add_to_cart),0) AS begin_checkout_sim,
ROUND(end_checkout + (product_view - add_to_cart)*0.2*(1.0*end_checkout/begin_checkout)*(1.0*begin_checkout/add_to_cart),0) AS end_checkout_sim,
ROUND(purchase + (product_view - add_to_cart)*0.2*(1.0*end_checkout/begin_checkout)*(1.0*begin_checkout/add_to_cart)*(1.0*purchase/end_checkout),0) AS purchase_sim,
ROUND(purchase + (product_view - add_to_cart)*0.2*(1.0*end_checkout/begin_checkout)*(1.0*begin_checkout/add_to_cart)*(1.0*purchase/end_checkout),0) - purchase AS incremento_purchase_users,
35*(ROUND(purchase + (product_view - add_to_cart)*0.2*(1.0*end_checkout/begin_checkout)*(1.0*begin_checkout/add_to_cart)*(1.0*purchase/end_checkout),0) - purchase) AS ganacia_sim_amount
FROM baseline

)


SELECT * FROM simulated_funnel

In [ ]:
%%sql


SELECT a.*,
       b.*

 FROM ecommerce_events AS a INNER JOIN dim_products AS b ON a.sku = b.sku
 WHERE a.event_name = 'purchase'
